In [ ]:
# import all the things!
import os
import pandas as pd
import numpy as np
import pickle

In [ ]:
CURVES_DIR = "../../data/ccCurvesOGPos"
origin_pts = {BUSH_NUM: {i: None for i in range(1, 4)} for BUSH_NUM in [1, 3, 5, 9, 14, 23]}
BUSH_NUMS = [1, 3, 5, 9, 14, 23]

for BUSH_NUM in BUSH_NUMS:
    CURVE_FILES = [os.path.join(CURVES_DIR, f"B{BUSH_NUM}_branch{i}_smoothpolyline.txt") for i in range(1, 4)]
    curves = {i: None for i in range(1, 4)}
    bottom_pts = {i: None for i in range(1, 4)}
    for i, curve_file in enumerate(CURVE_FILES, start=1):
        curve_df = pd.read_csv(curve_file, delimiter=',', header=None)
        # print(curve_df)
        og_array = curve_df.to_numpy()
        trans_array = np.zeros_like(og_array)
        trans_array[:,0] = og_array[:,2].copy() # +x in mujoco is +z in camera ("forwards")
        trans_array[:,1] = -og_array[:,0].copy() # +y in mujoco is +x in camera ("right")
        trans_array[:,2] = -og_array[:,1].copy() # +z in mujoco is +y in camera ("up")
        curves[i] = trans_array/1000
        bottom_pts[i] = curves[i][-1]

    # Make the lowest height the origin (z=0) for each curve
    bottom_zs = {i: bottom_pts[i][2] for i in range(1, 4)}
    absolute_bottom = min(bottom_zs.values())
    average_xs = np.mean([bottom_pts[i][0] for i in range(1, 4)])
    print(absolute_bottom)
    for i in range(1, 4):
        curves[i][:,2] -= absolute_bottom
        curves[i][:,0] -= average_xs
        bottom_pts[i] = curves[i][-1]
        row = [BUSH_NUM, i] + bottom_pts[i].tolist()
        origin_pts[BUSH_NUM][i] = bottom_pts[i]

# pickle origin points dict
pickle_file = os.path.join(CURVES_DIR, "origin_pts.pkl")
with open(pickle_file, 'wb') as f:
    pickle.dump(origin_pts, f)  

-0.246
-0.266
-0.299
-0.29
-0.228
0.46280004883


0.0
[-0.01833333  0.1641429   0.        ]
[0.00266667 0.034      0.094     ]
[ 0.01566667 -0.003       0.035     ]
